# analyse processing time for different fft sizes

In [15]:
import re
from collections import defaultdict

def parse_and_stats(filename):
    times_by_n = defaultdict(list)
    pattern = re.compile(r"processFrame:\s+([\d.]+)\s+ms\s+\(N=(\d+)\)")

    with open(filename) as f:
        for line in f:
            m = pattern.search(line)
            if m:
                times_by_n[int(m.group(2))].append(float(m.group(1)))

    for n in sorted(times_by_n):
        vals = times_by_n[n]
        count = len(vals)
        mean = sum(vals) / count
        std = (sum((v - mean) ** 2 for v in vals) / count) ** 0.5
        p95 = sorted(vals)[int(count * 0.95)]
        p95std = p95 + std
        print(f"N={n:6d}  n={count:4d}  mean={mean:.3f} ms  std={std:.3f} ms  p95={p95:.3f} ms  max={max(vals):.3f} ms  p95+std={p95std:.3f} ms")

In [16]:
from pathlib import Path

benchmark_dir = Path.home() / "Desktop" / "fdverb_runtime_measurements"

files_by_blocksize = defaultdict(list)
for f in sorted(benchmark_dir.glob("*_blocksize_*.txt")):
    m = re.search(r"blocksize_(\d+)", f.name)
    if m:
        files_by_blocksize[int(m.group(1))].append(f)

for blocksize in sorted(files_by_blocksize):
    files = files_by_blocksize[blocksize]
    lines = []
    for f in files:
        with open(f) as fh:
            lines.extend(fh.readlines())
    tmp = "/tmp/trimmed.txt"
    with open(tmp, "w") as fh:
        fh.writelines(lines[-1000:])
    label = f"blocksize {blocksize}" + (f"  ({len(files)} files)" if len(files) > 1 else "")
    print(f"=== {label} ===")
    parse_and_stats(tmp)
    print()